In [2]:
# =====================================================================
# BLOQUE 1 — Carga de datos
# =====================================================================
import numpy as np
import pandas as pd

RAW = "../data/raw/"
PROC = "../data/processed/"

features_df = pd.concat([
    pd.read_parquet(PROC + "features_dataset_part1.parquet"),
    pd.read_parquet(PROC + "features_dataset_part2.parquet")
], ignore_index=True)

df_maint = pd.read_csv(RAW + "PdM_maint.csv", parse_dates=["datetime"])
df_errors = pd.read_csv(RAW + "PdM_errors.csv", parse_dates=["datetime"])
df_failures = pd.read_csv(RAW + "PdM_failures.csv", parse_dates=["datetime"])

# Misma resolución de fecha en todas las tablas (evita error en merge_asof)
for df in [features_df, df_maint, df_errors, df_failures]:
    df["datetime"] = df["datetime"].astype("datetime64[ns]")

# Orden por máquina y hora (necesario para rolling y merge_asof)
features_df = features_df.sort_values(["machineID", "datetime"]).reset_index(drop=True)

print("=" * 60)
print("features_df:", features_df.shape)
print("maint:", df_maint.shape, "| errors:", df_errors.shape, "| failures:", df_failures.shape)
print("Registros por máquina:", features_df.groupby("machineID").size().unique())

features_df: (876100, 49)
maint: (3286, 3) | errors: (3919, 3) | failures: (761, 3)
Registros por máquina: [8761]


In [3]:
# =====================================================================
# BLOQUE 2 — Eliminar variables duplicadas
# =====================================================================
# days_since_maintenance = hours_since_maintenance / 24
# time_since_last_component_replacement_h = alias de hours_since_maintenance
print("=" * 60)
print("Correlación days vs hours:",
      round(features_df["days_since_maintenance"].corr(features_df["hours_since_maintenance"]), 4))
print("Alias idéntico:",
      (features_df["time_since_last_component_replacement_h"] == features_df["hours_since_maintenance"]).all())

cols_duplicadas = ["days_since_maintenance", "time_since_last_component_replacement_h"]
features_v2 = features_df.drop(columns=cols_duplicadas).copy()
print("features_v2:", features_v2.shape)

Correlación days vs hours: 1.0
Alias idéntico: True
features_v2: (876100, 47)


In [4]:
# =====================================================================
# BLOQUE 3 — Variables nuevas (solo información pasada, sin leakage)
# =====================================================================
sensores = ["volt", "rotate", "pressure", "vibration"]

# --- 3.1 Horas desde el último reemplazo de cada componente (comp1..comp4)
# merge_asof backward: toma el último mantenimiento con datetime <= hora actual
for comp in ["comp1", "comp2", "comp3", "comp4"]:
    m = (df_maint[df_maint["comp"] == comp][["machineID", "datetime"]]
         .rename(columns={"datetime": f"last_{comp}"})
         .sort_values(f"last_{comp}"))
    features_v2 = pd.merge_asof(
        features_v2.sort_values("datetime"), m,
        left_on="datetime", right_on=f"last_{comp}",
        by="machineID", direction="backward"
    )
    features_v2[f"hours_since_{comp}"] = (
        (features_v2["datetime"] - features_v2[f"last_{comp}"]).dt.total_seconds() / 3600
    ).fillna(8760)  # sin historial: mismo criterio de imputación del notebook 04
    features_v2 = features_v2.drop(columns=f"last_{comp}")

features_v2 = features_v2.sort_values(["machineID", "datetime"]).reset_index(drop=True)

# --- 3.2 Conteo de errores por tipo en 24h y 7d
err_hora = (df_errors.assign(n=1)
            .pivot_table(index=["machineID", "datetime"], columns="errorID",
                         values="n", aggfunc="sum", fill_value=0)
            .reset_index())
features_v2 = features_v2.merge(err_hora, on=["machineID", "datetime"], how="left")
tipos_error = [c for c in err_hora.columns if c.startswith("error")]
features_v2[tipos_error] = features_v2[tipos_error].fillna(0)

g = features_v2.groupby("machineID")
for e in tipos_error:
    features_v2[f"{e}_last_24h"] = g[e].transform(lambda s: s.rolling(24, min_periods=1).sum())
    features_v2[f"{e}_last_7d"] = g[e].transform(lambda s: s.rolling(168, min_periods=1).sum())
features_v2 = features_v2.drop(columns=tipos_error)

# --- 3.3 Rolling de ventanas largas (72h y 168h)
for s in sensores:
    features_v2[f"{s}_roll_mean_72h"] = g[s].transform(lambda x: x.rolling(72, min_periods=1).mean())
    features_v2[f"{s}_roll_mean_168h"] = g[s].transform(lambda x: x.rolling(168, min_periods=1).mean())
    features_v2[f"{s}_roll_std_168h"] = g[s].transform(lambda x: x.rolling(168, min_periods=2).std())

# --- 3.4 Pendiente de tendencia en 24h (regresión lineal móvil)
# slope = (mean(t·x) - mean(t)·mean(x)) / (mean(t²) - mean(t)²)
features_v2["_t"] = g.cumcount().astype(float)
roll = lambda col: features_v2.groupby("machineID")[col].transform(lambda x: x.rolling(24, min_periods=2).mean())
mt = roll("_t")
var_t = features_v2.assign(_t2=features_v2["_t"] ** 2).groupby("machineID")["_t2"] \
    .transform(lambda x: x.rolling(24, min_periods=2).mean()) - mt ** 2
for s in sensores:
    features_v2["_tx"] = features_v2["_t"] * features_v2[s]
    features_v2[f"{s}_slope_24h"] = (roll("_tx") - mt * features_v2[f"{s}_roll_mean_24h"]) / var_t
features_v2 = features_v2.drop(columns=["_t", "_tx"])

# Primeras horas de cada máquina sin ventana suficiente
features_v2 = features_v2.fillna(0)

# --- Validaciones
print("=" * 60)
print("features_v2:", features_v2.shape)
print("Nulos:", features_v2.isna().sum().sum(),
      "| Infinitos:", np.isinf(features_v2.select_dtypes("number")).sum().sum())
nuevas = [c for c in features_v2.columns if c not in features_df.columns]
print(f"Variables nuevas ({len(nuevas)}):", nuevas)

# --- Exportar en 4 partes, float32 (límite 100 MB por archivo en GitHub)
export_df = features_v2.copy()
cols_float = export_df.select_dtypes("float64").columns
export_df[cols_float] = export_df[cols_float].astype("float32")
for i, (ini, fin) in enumerate([(1, 25), (26, 50), (51, 75), (76, 100)], start=1):
    export_df[export_df["machineID"].between(ini, fin)].to_parquet(
        PROC + f"features_dataset_v2_part{i}.parquet", compression="brotli", index=False)
print("Exportado: features_dataset_v2_part1..part4.parquet")

features_v2: (876100, 77)
Nulos: 0 | Infinitos: 0
Variables nuevas (30): ['hours_since_comp1', 'hours_since_comp2', 'hours_since_comp3', 'hours_since_comp4', 'error1_last_24h', 'error1_last_7d', 'error2_last_24h', 'error2_last_7d', 'error3_last_24h', 'error3_last_7d', 'error4_last_24h', 'error4_last_7d', 'error5_last_24h', 'error5_last_7d', 'volt_roll_mean_72h', 'volt_roll_mean_168h', 'volt_roll_std_168h', 'rotate_roll_mean_72h', 'rotate_roll_mean_168h', 'rotate_roll_std_168h', 'pressure_roll_mean_72h', 'pressure_roll_mean_168h', 'pressure_roll_std_168h', 'vibration_roll_mean_72h', 'vibration_roll_mean_168h', 'vibration_roll_std_168h', 'volt_slope_24h', 'rotate_slope_24h', 'pressure_slope_24h', 'vibration_slope_24h']
Exportado: features_dataset_v2_part1..part4.parquet


In [5]:
# =====================================================================
# BLOQUE 4 — Ablación (mismo Random Forest y split del notebook 05)
# =====================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

# --- Target a 48h: falla en (t, t+48h], mismo criterio que failure_next_24h
fal = df_failures[["machineID", "datetime"]].drop_duplicates() \
    .rename(columns={"datetime": "next_failure"}).sort_values("next_failure")
tmp = pd.merge_asof(
    features_v2[["machineID", "datetime"]].reset_index().sort_values("datetime"), fal,
    left_on="datetime", right_on="next_failure", by="machineID",
    direction="forward", allow_exact_matches=False
).set_index("index").sort_index()
horas_a_falla = (tmp["next_failure"] - tmp["datetime"]).dt.total_seconds() / 3600
features_v2["failure_next_48h"] = (horas_a_falla <= 48).astype(int)

check_24 = (horas_a_falla <= 24).astype(int)
print("=" * 60)
print("Coincidencia target 24h reconstruido vs original:",
      round((check_24 == features_v2["failure_next_24h"]).mean(), 5))
print("Positivos 24h:", round(features_v2["failure_next_24h"].mean() * 100, 2), "%",
      "| Positivos 48h:", round(features_v2["failure_next_48h"].mean() * 100, 2), "%")

# --- Grupos de variables
targets = ["failure_next_24h", "failure_next_48h"]
ids = ["datetime", "machineID"]
cols_originales = [c for c in features_df.columns if c not in ids + ["failure_next_24h"]]
cols_v2 = [c for c in features_v2.columns if c not in ids + targets]
cols_error = [c for c in cols_v2 if "error" in c]
cols_sin_error = [c for c in cols_v2 if c not in cols_error]
cols_solo_sensores = [c for c in cols_sin_error
                      if any(c.startswith(s) for s in ["volt", "rotate", "pressure", "vibration"])]

# --- Split temporal del notebook 05 (gap de 24h)
train = features_v2["datetime"] <= "2015-09-30 23:00:00"
test = features_v2["datetime"] >= "2015-10-02 00:00:00"
datos = features_v2.merge(features_df[ids + ["days_since_maintenance",
                                             "time_since_last_component_replacement_h"]], on=ids)

def evaluar(cols, target):
    """Entrena el RF del baseline y devuelve métricas en test. Returns: dict"""
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=20,
                                class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(datos.loc[train, cols], datos.loc[train, target])
    p = rf.predict_proba(datos.loc[test, cols])[:, 1]
    y = datos.loc[test, target]
    return {"n_features": len(cols),
            "PR_AUC": round(average_precision_score(y, p), 4),
            "ROC_AUC": round(roc_auc_score(y, p), 4),
            "Precision@0.5": round(precision_score(y, p >= 0.5), 4),
            "Recall@0.5": round(recall_score(y, p >= 0.5), 4)}

escenarios = {
    "A. Original (46)":         (cols_originales, "failure_next_24h"),
    "B. v2 completo":           (cols_v2, "failure_next_24h"),
    "C. v2 sin errores":        (cols_sin_error, "failure_next_24h"),
    "D. Solo sensores":         (cols_solo_sensores, "failure_next_24h"),
    "E. v2 completo · 48h":     (cols_v2, "failure_next_48h"),
    "F. v2 sin errores · 48h":  (cols_sin_error, "failure_next_48h"),
}

resultados = []
for nombre, (cols, target) in escenarios.items():
    print(f"Entrenando {nombre}...")
    resultados.append({"escenario": nombre, "target": target, **evaluar(cols, target)})

ablacion_df = pd.DataFrame(resultados)
print("=" * 60)
print(ablacion_df.to_string(index=False))

Coincidencia target 24h reconstruido vs original: 1.0
Positivos 24h: 1.96 % | Positivos 48h: 3.88 %
Entrenando A. Original (46)...
Entrenando B. v2 completo...
Entrenando C. v2 sin errores...
Entrenando D. Solo sensores...
Entrenando E. v2 completo · 48h...
Entrenando F. v2 sin errores · 48h...
              escenario           target  n_features  PR_AUC  ROC_AUC  Precision@0.5  Recall@0.5
       A. Original (46) failure_next_24h          46  0.9929   0.9999         0.8070      0.9990
         B. v2 completo failure_next_24h          74  0.9998   1.0000         0.9514      0.9990
      C. v2 sin errores failure_next_24h          59  0.9022   0.9986         0.6053      0.9937
       D. Solo sensores failure_next_24h          48  0.2817   0.9700         0.1909      0.9736
   E. v2 completo · 48h failure_next_48h          74  0.9468   0.9969         0.6301      0.9603
F. v2 sin errores · 48h failure_next_48h          59  0.8594   0.9950         0.5873      0.9661
